# HydrAI: Plug Flow Reactor Simulation

**HydrAI** = **Hydr**ocarbon + **AI** (Machine Learning)

This notebook provides an interactive interface for running PFR simulations using Cantera.

## Features
- Multi-reactant support (ethane, propane, naphtha, n-hexane)
- Interactive configuration
- Real-time simulation progress
- Comprehensive visualization
- Results export

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/ 
# (our package) instead of the actual cantera library when pfr_simulator.py imports it
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Get project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
sys.path.insert(0, str(project_root))
os.chdir(project_root)  # so configs/, mechanisms/, data/ resolve correctly
print(f"Project root: {project_root}")

from src.cantera.pfr_simulator import (
    load_reactant_database,
    generate_config_for_reactant,
    setup_mechanism,
    setup_initial_conditions,
    setup_heat_flux,
    run_simulation,
    process_and_visualize_results
)

import numpy as np
import matplotlib.pyplot as plt

from src.utils.plot_style import (
    load_aesthetics,
    apply_style,
    get_profile_style,
    setup_axes,
    setup_legend,
)

# Figure export controls
IF_SAVE_PLOTS = True
PLOT_EXPORT_DIR = project_root / "outputs" / "figures" / "Main_1_run_pfr"

figure_aesthetics = load_aesthetics()
apply_style(figure_aesthetics)

print("HydrAI: PFR Simulation System")
print("=" * 60)
print(f"Save plots: {IF_SAVE_PLOTS}  |  Plot dir: {PLOT_EXPORT_DIR}")

## Step 1: Select Reactant

Choose the reactant you want to simulate. Available options are listed below.

In [ ]:
# Load reactant database
database = load_reactant_database()

# Display available reactants
print("Available Reactants:")
print("-" * 60)
for key, info in database['reactants'].items():
    print(f"  {key:12s}: {info['name']:15s} - {info['description']}")
print("-" * 60)

# Set reactant here (change this to select different reactant)
REACTANT_KEY = 'n-hexane'  # Options: 'ethane', 'propane', 'naphtha', 'n-hexane'

if REACTANT_KEY not in database['reactants']:
    print(f"\n[ERROR] Reactant '{REACTANT_KEY}' not found!")
    print(f"Available reactants: {list(database['reactants'].keys())}")
else:
    reactant_info = database['reactants'][REACTANT_KEY]
    print(f"\n[OK] Selected: {reactant_info['name']} ({REACTANT_KEY})")
    print(f"    Description: {reactant_info['description']}")

## Step 2: Load Configuration

Generate the simulation configuration for the selected reactant.

In [ ]:
# Generate configuration
config = generate_config_for_reactant(REACTANT_KEY, database)
reactant_info = database['reactants'][REACTANT_KEY]

print(f"Loaded configuration for: {config['simulation_info']['title']}")
print(f"Version: {config['simulation_info']['version']}")
print(f"\nConfiguration Summary:")
print(f"  - Mechanism: {config['mechanism']['reaction_mechanism_file']}")
print(f"  - Initial Temperature: {config['initial_conditions']['temperature_K']} K")
print(f"  - Initial Pressure: {config['initial_conditions']['pressure_Pa']/1e5:.2f} bar")
print(f"  - Reactor Length: {config['reactor_geometry']['length_m']} m")
print(f"  - Reactor Diameter: {config['reactor_geometry']['diameter_m']*1000:.1f} mm")
print(f"  - Mass Flow Rate: {config['operating_conditions']['mass_flow_rate_kgps']} kg/s")
print(f"  - Number of Steps: {config['simulation_settings']['n_steps']}")

## Step 3: Setup Mechanism and Initial Conditions

Initialize the Cantera mechanism and set up initial conditions.

In [ ]:
# Setup mechanism
print("Setting up mechanism...")
gas = setup_mechanism(config)
print(f"[OK] Mechanism loaded: {len(gas.species_names)} species, {len(gas.reaction_equations())} reactions")

# Setup initial conditions
print("\nSetting up initial conditions...")
T_0, p_0, composition_0, length, diameter, area, roughness, mass_flow_rate, u_0 = \
    setup_initial_conditions(gas, config)

print(f"[OK] Initial conditions set:")
print(f"  - Temperature: {T_0:.1f} K")
print(f"  - Pressure: {p_0/1e5:.2f} bar")
print(f"  - Initial velocity: {u_0:.2f} m/s")
print(f"  - Composition: {composition_0}")

# Setup heat flux
print("\nSetting up heat flux profile...")
hf, z_profile, heatflux_profile = setup_heat_flux(config)
print(f"[OK] Heat flux profile loaded: {len(heatflux_profile)} data points")
print(f"  - Average heat flux: {np.mean(heatflux_profile)/1e3:.1f} kW/m²")

## Step 4: Run Simulation

Execute the PFR simulation. This may take a few minutes depending on the mechanism complexity.

In [ ]:
# Run simulation
print("=" * 60)
print("Starting PFR Simulation...")
print("=" * 60)

import time
start_time = time.time()

states1 = run_simulation(gas, config, reactant_info, hf, T_0, p_0, 
                        length, diameter, area, roughness, mass_flow_rate, u_0)

elapsed_time = time.time() - start_time
print(f"\n[OK] Simulation completed in {elapsed_time:.2f} seconds")
print(f"  - Number of points: {len(states1.z)}")
print(f"  - Final temperature: {states1.T[-1]:.1f} K")
print(f"  - Final pressure: {states1.P[-1]/1e5:.2f} bar")

## Step 5: Process Results and Visualize

Generate visualizations and calculate key performance metrics.

In [ ]:
# Process and visualize results
print("Processing results and generating visualizations...")
conversion, yields = process_and_visualize_results(gas, states1, config, 
                                                  reactant_info, hf, T_0, p_0, u_0)

print("[OK] Results processed and visualizations generated")

## Step 6: Results Summary

Display key simulation results and performance metrics.

In [ ]:
# Calculate residence time
residence_time = np.trapz(1/states1.velocity, states1.z)

# Display summary
print("=" * 60)
print("SIMULATION COMPLETED SUCCESSFULLY!")
print("=" * 60)
print(f"[OK] {reactant_info['name']} conversion: {conversion:.1f}%")
print(f"[OK] Temperature rise: {states1.T[-1] - T_0:.1f} K")
print(f"[OK] Pressure drop: {(p_0 - states1.P[-1])/1e5:.2f} bar")
print(f"[OK] Residence time: {residence_time:.3f} s")
print(f"\n[OK] Product Yields:")
for product, yield_val in yields.items():
    print(f"  - {product}: {yield_val:.2f}%")
print(f"\n[OK] Files generated:")
print(f"  - outputs/figures/*.png: Visualization plots")
print(f"  - outputs/results/*.csv: Simulation data")
print(f"  - outputs/results/*.dat: Simulation summary")
print("=" * 60)

## Quick Visualization

Display key profiles inline in the notebook.

In [ ]:
# Quick inline visualization (sizes, colors, grid from configs/style/figure_aesthetics.json)
w, h = tuple(figure_aesthetics['figure']['figsize'])
fig, axes = plt.subplots(2, 2, figsize=(w * 1.8, h * 10 / 6))

# Temperature profile
_st = get_profile_style('temperature', figure_aesthetics)
axes[0, 0].plot(
    states1.z, states1.T,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[0, 0].set_xlabel('Reactor Position [m]')
axes[0, 0].set_ylabel(_st['ylabel'])
axes[0, 0].set_title(_st['title'])
setup_axes(axes[0, 0], figure_aesthetics)

# Pressure profile
_st = get_profile_style('pressure', figure_aesthetics)
axes[0, 1].plot(
    states1.z, states1.P / 1e5,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[0, 1].set_xlabel('Reactor Position [m]')
axes[0, 1].set_ylabel(_st['ylabel'])
axes[0, 1].set_title(_st['title'])
setup_axes(axes[0, 1], figure_aesthetics)

# Velocity profile
_st = get_profile_style('velocity', figure_aesthetics)
axes[1, 0].plot(
    states1.z, states1.velocity,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[1, 0].set_xlabel('Reactor Position [m]')
axes[1, 0].set_ylabel(_st['ylabel'])
axes[1, 0].set_title(_st['title'])
setup_axes(axes[1, 0], figure_aesthetics)

# Reactant conversion
# Get species index - handle species name format (may have :1 suffix)
feed_species = reactant_info['feed_species']
# Remove :1 suffix if present (format from database vs mechanism)
species_name = feed_species.split(':')[0] if ':' in feed_species else feed_species

try:
    species_idx = gas.species_index(species_name)
except:
    # Fallback: search for species by partial match
    species_idx = None
    for i, species in enumerate(gas.species_names):
        if species_name in species or reactant_info['name'].lower() in species.lower():
            species_idx = i
            break
    if species_idx is None:
        raise ValueError(f"Could not find species: {feed_species} (tried: {species_name})")

# Access mass fractions from SolutionArray using 2D indexing: states1.Y[:, species_idx]
reactant_mass_frac = states1.Y[:, species_idx]
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100
_st = get_profile_style('conversion', figure_aesthetics)
axes[1, 1].plot(
    states1.z, conversion_profile,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[1, 1].set_xlabel('Reactor Position [m]')
axes[1, 1].set_ylabel('Conversion [%]')
axes[1, 1].set_title(f'{reactant_info["name"]} Conversion Profile')
setup_axes(axes[1, 1], figure_aesthetics)

plt.tight_layout()
if IF_SAVE_PLOTS:
    PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOT_EXPORT_DIR / f"main1_quick_profiles_{REACTANT_KEY}.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n[OK] Inline visualizations displayed above")

In [ ]:
# Combined Reactant Conversion and Product Formation
font_cfg = figure_aesthetics['font']
w, h = tuple(figure_aesthetics['figure']['figsize'])
fig, ax = plt.subplots(figsize=(w * 1.2, h * 7 / 6))

# Get reactant conversion profile
feed_species = reactant_info['feed_species']
species_name = feed_species.split(':')[0] if ':' in feed_species else feed_species

try:
    species_idx = gas.species_index(species_name)
except:
    # Fallback: search for species by partial match
    species_idx = None
    for i, species in enumerate(gas.species_names):
        if species_name in species or reactant_info['name'].lower() in species.lower():
            species_idx = i
            break
    if species_idx is None:
        raise ValueError(f"Could not find species: {feed_species}")

# Calculate reactant conversion
reactant_mass_frac = states1.Y[:, species_idx]
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100

# Plot reactant conversion (increasing) or reactant remaining (decreasing)
# Option 1: Show conversion (0% to 100% as reactant is consumed)
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100
# Option 2: Show reactant remaining (100% to 0% as reactant is consumed)
reactant_remaining = (reactant_mass_frac / reactant_mass_frac[0]) * 100

_st_rem = get_profile_style('temperature', figure_aesthetics)
prod_style = get_profile_style('products', figure_aesthetics)
prod_colors = prod_style.get('colors') or [prod_style['color']]

# Plot reactant remaining (decreasing) - this shows reactant disappearing
ax.plot(
    states1.z, reactant_remaining,
    color=_st_rem['color'], linewidth=_st_rem['linewidth'],
    linestyle=_st_rem['linestyle'], alpha=_st_rem['alpha'],
    label=f'{reactant_info["name"]} Remaining',
)

# Get target products and plot their formation
target_products = reactant_info.get('target_products', [])
product_names = reactant_info.get('product_names', [])

# Calculate and plot product yields (as percentage of initial reactant)
initial_reactant_mass = reactant_mass_frac[0]
_ln = figure_aesthetics['line']
for i, product in enumerate(target_products):
    try:
        # Handle product name format (may have :1 suffix)
        product_name_clean = product.split(':')[0] if ':' in product else product
        product_idx = gas.species_index(product_name_clean)

        # Get product mass fraction profile
        product_mass_frac = states1.Y[:, product_idx]

        # Calculate product yield as percentage of initial reactant
        # Product yield = (product mass fraction / initial reactant mass fraction) * 100
        # This shows how much product is formed relative to initial reactant
        product_yield = (product_mass_frac / initial_reactant_mass) * 100

        product_name = product_names[i] if i < len(product_names) else product_name_clean
        ax.plot(
            states1.z, product_yield,
            color=prod_colors[i % len(prod_colors)],
            linewidth=_ln['width'],
            linestyle=_ln['style'],
            alpha=_ln['alpha'],
            label=product_name,
        )
    except:
        continue

ax.set_xlabel('Reactor Position [m]')
ax.set_ylabel('Reactant Remaining / Product Yield [%]')
ax.set_title(
    f'{reactant_info["name"]} Consumption and Products Formation',
    fontsize=font_cfg['title_size'],
    fontweight=font_cfg['title_weight'],
)
setup_legend(ax, figure_aesthetics, ncol=4)
setup_axes(ax, figure_aesthetics)
ax.set_ylim(bottom=0)

plt.tight_layout()
if IF_SAVE_PLOTS:
    PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOT_EXPORT_DIR / f"main1_conversion_products_{REACTANT_KEY}.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n[OK] Combined conversion and product formation plot displayed above")